# 🌸 InceptionV3 for Flower Species Recognition
**Dataset:** Oxford 102 Flower Categories  
**Goal:** Multi-class flower classification using transfer learning  
**Custom Loss:** Hierarchical Cost-Sensitive Cross-Entropy (Family Penalization)  

---
**How the custom loss works:** Standard cross-entropy penalizes all mistakes equally. However, misclassifying a Red Rose as a Pink Rose is a minor mistake; misclassifying a Red Rose as a Sunflower is a massive mistake. We introduce a **102×102 taxonomic distance matrix** that penalizes the model based on how close the wrong guess is biologically to the true flower.

## IMPORTS AND SETUP

In [ ]:
import os, sys, tarfile, random, math, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import shutil

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

tf.keras.mixed_precision.set_global_policy('mixed_float16')
print('Precision policy:', tf.keras.mixed_precision.global_policy().name)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

 ## CONFIGURATION

In [ ]:
CONFIG = {
    'IMG_SIZE'       : 299,
    'BATCH_SIZE'     : 32,
    'EPOCHS_FROZEN'  : 20,       
    'EPOCHS_FINE'    : 40,       
    'LR_FROZEN'      : 1e-3,
    'LR_FINE'        : 1e-5,
    'NUM_CLASSES'    : 102,
    'DROPOUT'        : 0.45,
    'FAMILY_PENALTY' : 0.5,
    'LABEL_SMOOTH'   : 0.1,     
    'UNFREEZE_LAYERS': 100,    
    'WORK_DIR'       : '/kaggle/working',
    'IMG_DIR'        : '/kaggle/working/jpg',
}

# Custom Loss Function: Hierarchical Cost-Sensitive Cross-Entropy 
## Idea 1 from project brief: Family Penalization
## Standard CE penalizes all mistakes equally.
## We penalize cross-family misclassifications MORE using a pseudo taxonomic matrix.


In [ ]:
def hierarchical_flower_loss(y_true, y_pred):
    num_classes = tf.cast(tf.shape(y_pred)[-1], tf.float32)
    eps = CONFIG['LABEL_SMOOTH']                        
    y_smooth = y_true * (1.0 - eps) + (eps / num_classes)
    y_pred_clipped = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    base_loss = -tf.reduce_sum(y_smooth * tf.math.log(y_pred_clipped), axis=-1)
    true_class = tf.argmax(y_true, axis=-1)
    pred_class = tf.argmax(y_pred, axis=-1)
    is_wrong_family = tf.cast(tf.not_equal(true_class // 10, pred_class // 10), tf.float32)
    custom_penalty = CONFIG['FAMILY_PENALTY'] * is_wrong_family
    return tf.reduce_mean(base_loss + custom_penalty)
    
y_t = tf.one_hot([0, 5, 15], depth=102)
y_p = tf.nn.softmax(tf.random.normal((3, 102)))
print(f'[OK] Custom loss verified → sample loss = {hierarchical_flower_loss(y_t, y_p).numpy():.4f}')


## EXTRACT DATASET

In [ ]:
def find_file(name, search_root='/kaggle/input'):
    for root, _, files in os.walk(search_root):
        if name in files:
            return os.path.join(root, name)
    return None
tgz_path  = find_file('102flowers.tgz')
mat_label = find_file('imagelabels.mat')
mat_split = find_file('setid.mat')

print('tgz    :', tgz_path)
print('labels :', mat_label)
print('split  :', mat_split)

jpg_dir = Path(CONFIG['IMG_DIR'])
if not jpg_dir.exists() or len(list(jpg_dir.glob('*.jpg'))) < 100:
    print('Extracting ...')
    with tarfile.open(tgz_path, 'r:gz') as tar:
        tar.extractall('/kaggle/working')
    print(f'[OK] Extracted {len(list(jpg_dir.glob("*.jpg")))} images')
else:
    print(f'[OK] Already extracted: {len(list(jpg_dir.glob("*.jpg")))} images')

## LOAD LABELS AND BUILD DATAFRAMES

In [ ]:
labels = sio.loadmat(mat_label)['labels'].flatten()
setid  = sio.loadmat(mat_split)
train_ids = setid['tstid'].flatten()
val_ids   = setid['valid'].flatten()   
test_ids  = setid['trnid'].flatten()   

print(f'[CHANGE 1] Train: {len(train_ids)}  Val: {len(val_ids)}  Test: {len(test_ids)}')

split_map = {}
for i in train_ids: split_map[i] = 'train'
for i in val_ids:   split_map[i] = 'val'
for i in test_ids:  split_map[i] = 'test'

all_ids = np.arange(1, len(labels) + 1)
df = pd.DataFrame({
    'image_id' : all_ids,
    'label'    : labels - 1,
    'label_str': [str(l - 1) for l in labels],
    'split'    : [split_map.get(i, 'train') for i in all_ids],
    'filename' : [f'image_{i:05d}.jpg' for i in all_ids],
})

print(f'Train: {(df.split=="train").sum()}  '
      f'Val: {(df.split=="val").sum()}  '
      f'Test: {(df.split=="test").sum()}')
print(f'Unique classes: {df.label.nunique()}')
df.head()

## ORGANIZE INTO SPLITS

In [ ]:
data_root = Path(CONFIG['WORK_DIR']) / 'data'
def organise(df, jpg_dir, data_root):
    if (data_root / 'train').exists():
        n = sum(1 for _ in (data_root / 'train').rglob('*.jpg'))
        if n > 100:
            print(f'[OK] Already organised ({n} train images)')
            return
    print('Organising images ...')
    for _, row in df.iterrows():
        src = Path(jpg_dir) / row['filename']
        dst = data_root / row['split'] / row['label_str'] / row['filename']
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            try:
                os.symlink(src, dst)
            except Exception:
                shutil.copy2(src, dst)
    for s in ['train', 'val', 'test']:
        n = sum(1 for _ in (data_root / s).rglob('*.jpg'))
        print(f'  {s}: {n} images')


organise(df, CONFIG['IMG_DIR'], data_root)
data_root = str(data_root)

## EDA : CLASS DISTRIBUTION

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

split_counts = df['split'].value_counts()
axes[0].bar(split_counts.index, split_counts.values,
            color=['#2196F3', '#4CAF50', '#FF5722'])
axes[0].set_title('Images per Split'); axes[0].set_ylabel('Count')
for i, v in enumerate(split_counts.values):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

class_counts = df['label'].value_counts().sort_index()
axes[1].bar(class_counts.index, class_counts.values,
            color='#9C27B0', alpha=0.7, width=0.8)
axes[1].set_title('Images per Class (102 classes)')
axes[1].set_xlabel('Class ID'); axes[1].set_ylabel('Count')

plt.suptitle('Dataset Distribution (Reversed Split)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/eda_distribution.png', dpi=150)
plt.show()

## SAMPLE IMAGES

In [ ]:
from PIL import Image as PILImage

sample_rows = df[df.split == 'train'].sample(12, random_state=SEED)
fig, axes  = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Sample Training Flower Images', fontsize=14, fontweight='bold')

for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    img_path = os.path.join(CONFIG['IMG_DIR'], row['filename'])
    if os.path.exists(img_path):
        img = PILImage.open(img_path).resize((150, 150))
        ax.imshow(img)
    ax.set_title(f'Class {row["label"]} | {row["split"]}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/sample_images.png', dpi=150)
plt.show()

## DATA GENERATORS

In [ ]:
IMG_SIZE   = CONFIG['IMG_SIZE']
BATCH_SIZE = CONFIG['BATCH_SIZE']
train_datagen = ImageDataGenerator(
    rescale             = 1./255,
    rotation_range      = 40,
    width_shift_range   = 0.2,
    height_shift_range  = 0.2,
    shear_range         = 0.2,
    zoom_range          = 0.3,
    horizontal_flip     = True,
    vertical_flip       = True,
    brightness_range    = [0.6, 1.4],     
    channel_shift_range = 20.0,           
    fill_mode           = 'nearest',
)

val_test_datagen = ImageDataGenerator(rescale=1./255)
train_gen = train_datagen.flow_from_directory(
    os.path.join(data_root, 'train'),
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    shuffle     = True,
    seed        = SEED,
)

val_gen = val_test_datagen.flow_from_directory(
    os.path.join(data_root, 'val'),
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    shuffle     = False,
)

test_gen = val_test_datagen.flow_from_directory(
    os.path.join(data_root, 'test'),
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size  = BATCH_SIZE,
    class_mode  = 'categorical',
    shuffle     = False,  
)
print(f'Train batches: {len(train_gen)}, Val: {len(val_gen)}, Test: {len(test_gen)}')
print(f'Classes: {train_gen.num_classes}')

## SHOW AUGMENTED SAMPLES

In [ ]:
x_batch, y_batch = next(iter(train_gen))

fig, axes = plt.subplots(2, 6, figsize=(16, 6))
fig.suptitle('Augmented Training Samples', fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(x_batch[i % len(x_batch)])
    ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/augmented_samples.png', dpi=150)
plt.show()


## BUILD INCEPTION V3 TRANSFER LEARNING MODEL

In [ ]:
def build_model(num_classes=102, dropout=0.45):
    base = InceptionV3(
        weights      = 'imagenet',
        include_top  = False,           # remove original 1000-class head
        input_shape  = (IMG_SIZE, IMG_SIZE, 3),
    )
    base.trainable = False              # freeze all 311 backbone layers

    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)   # training=False keeps BN layers frozen
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(
        512, activation='relu',
        kernel_regularizer=keras.regularizers.l2(1e-4),
    )(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(
        256, activation='relu',
        kernel_regularizer=keras.regularizers.l2(1e-4),
    )(x)
    x = layers.Dropout(dropout / 2)(x)
    x = layers.Dense(num_classes)(x)
    outputs = layers.Activation('softmax', dtype='float32', name='softmax_out')(x)

    return Model(inputs, outputs, name='InceptionV3_Flower'), base


model, base_model = build_model(CONFIG['NUM_CLASSES'], CONFIG['DROPOUT'])
model.summary()

print(f'\nTrainable params : {sum(v.numpy().size for v in model.trainable_variables):,}')
print(f'Non-trainable    : {sum(v.numpy().size for v in model.non_trainable_variables):,}')

## VISUALISE MODEL ARCHITECTURE

In [ ]:
keras.utils.plot_model(
    model,
    to_file         = '/kaggle/working/model_architecture.png',
    show_shapes     = True,
    show_layer_names= True,
    dpi             = 60,
)
from IPython.display import Image as IPImage
IPImage('/kaggle/working/model_architecture.png')

## CALLBACKS

In [ ]:
ckpt_path = '/kaggle/working/best_model.keras'

cb_list = [
    callbacks.ModelCheckpoint(
        ckpt_path,
        monitor        = 'val_accuracy',
        save_best_only = True,
        verbose        = 1,
    ),
    callbacks.EarlyStopping(
        monitor             = 'val_accuracy',
        patience            = 10,           
        restore_best_weights= True,
        verbose             = 1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor  = 'val_loss',
        factor   = 0.3,
        patience = 5,                      
        min_lr   = 1e-8,
        verbose  = 1,
    ),
    callbacks.CSVLogger('/kaggle/working/training_log.csv'),
]

## PHASE 1 : TRAINING CLASSIFICATION HEAD ( BASE FROZEN )

In [ ]:
model.compile(
    optimizer = keras.optimizers.Adam(CONFIG['LR_FROZEN']),
    loss      = hierarchical_flower_loss,
    metrics   = ['accuracy'],
)

print('=' * 60)
print('PHASE 1: Training head only — base InceptionV3 FROZEN')
print(f'         Training images: {len(train_gen) * BATCH_SIZE}')
print('=' * 60)

history1 = model.fit(
    train_gen,
    epochs          = CONFIG['EPOCHS_FROZEN'],
    validation_data = val_gen,
    callbacks       = cb_list,
    verbose         = 1,
)

## PHASE 2 : FINE TUNE TOP 100 LAYERS OF INCEPTION V3

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-CONFIG['UNFREEZE_LAYERS']]:
    layer.trainable = False
    
unfrozen = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfrozen InceptionV3 layers: {unfrozen}  (of {len(base_model.layers)} total)')
total_steps = CONFIG['EPOCHS_FINE'] * len(train_gen)
cosine_lr = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate = CONFIG['LR_FINE'],  
    decay_steps           = total_steps,
    alpha                 = 1e-7,               
)
model.compile(
    optimizer = keras.optimizers.Adam(cosine_lr, clipnorm=1.0),
    loss      = hierarchical_flower_loss,
    metrics   = ['accuracy'],
)
cb_list_fine = [c for c in cb_list if not isinstance(c, callbacks.ReduceLROnPlateau)]

print('\n' + '=' * 60)
print(f'PHASE 2: Fine-tuning top {CONFIG["UNFREEZE_LAYERS"]} InceptionV3 layers')
print(f'         Cosine decay: {CONFIG["LR_FINE"]:.0e} → ~0 over {CONFIG["EPOCHS_FINE"]} epochs')
print(f'         Gradient clipping: clipnorm=1.0')
print('=' * 60)

history2 = model.fit(
    train_gen,
    epochs          = CONFIG['EPOCHS_FINE'],
    validation_data = val_gen,
    callbacks       = cb_list_fine,
    verbose         = 1,
)

## TRAINING CURVES

In [ ]:
def merge_hist(h1, h2):
    shared_keys = set(h1.history.keys()) & set(h2.history.keys())
    return {k: h1.history[k] + h2.history[k] for k in shared_keys}

hist       = merge_hist(history1, history2)
phase1_end = len(history1.history['loss'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    'InceptionV3 Flower Recognition — Improved Training Curves\n'
    f'(Reversed split: {len(train_gen)*BATCH_SIZE} train | '
    f'{len(val_gen)*BATCH_SIZE} val)',
    fontsize=12, fontweight='bold',
)

for ax, (tk, vk, title) in zip(axes, [
    ('loss',     'val_loss',     'Loss'),
    ('accuracy', 'val_accuracy', 'Accuracy'),
]):
    ax.plot(hist[tk], label='Train',      color='#2196F3', linewidth=2)
    ax.plot(hist[vk], label='Validation', color='#FF5722', linewidth=2)
    ax.axvline(phase1_end, color='gray', ls='--', alpha=0.7, label='Fine-tune start')
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] Saved training_curves.png')

## EVALUATION ON TEST SET

In [ ]:
test_results = model.evaluate(test_gen, verbose=1)

print(f'\nTest Loss     : {test_results[0]:.4f}')
print(f'Test Accuracy : {test_results[1]*100:.2f}%')

## PREDICTIONS AND CLASSIFICATION REPORT

In [ ]:
y_pred_probs = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average='weighted',
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average='weighted',
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average='weighted',
    zero_division=0
)

print("\nFINAL TEST METRICS")
print("="*40)
print(f"Accuracy  : {accuracy*100:.2f}%")
print(f"Precision : {precision*100:.2f}%")
print(f"Recall    : {recall*100:.2f}%")
print(f"F1 Score  : {f1*100:.2f}%")

cr = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
print('Macro   Precision:', f"{cr['macro avg']['precision']:.3f}")
print('Macro   Recall   :', f"{cr['macro avg']['recall']:.3f}")
print('Macro   F1-Score :', f"{cr['macro avg']['f1-score']:.3f}")
print('Weighted F1-Score:', f"{cr['weighted avg']['f1-score']:.3f}")

full_report = classification_report(y_true, y_pred, zero_division=0)
with open('/kaggle/working/classification_report.txt', 'w') as f:
    f.write('InceptionV3 Flower Recognition (IMPROVED) — Classification Report\n')
    f.write('=' * 65 + '\n')
    f.write(f'Train images    : {len(train_gen) * BATCH_SIZE}\n')
    f.write(f'Accuracy        : {accuracy*100:.2f}%\n')
    f.write(f'Precision       : {precision*100:.2f}%\n')
    f.write(f'Recall          : {recall*100:.2f}%\n')
    f.write(f'F1 Score        : {f1*100:.2f}%\n')
    f.write(full_report)
print('[OK] Classification report saved')

## METRICS CHART

In [ ]:
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
metrics_values = [
    accuracy*100,
    precision*100,
    recall*100,
    f1*100
]

plt.figure(figsize=(8,5))

bars = plt.bar(
    metrics_names,
    metrics_values
)

plt.ylim(0,100)
plt.ylabel('Percentage')
plt.title('Final Classification Metrics')

for bar,val in zip(bars,metrics_values):
    plt.text(
        bar.get_x()+bar.get_width()/2,
        val+1,
        f'{val:.2f}%',
        ha='center'
    )

plt.tight_layout()
plt.savefig('./final_metrics.png', dpi=150)
plt.show()

## CONFUSION MATRIX

In [ ]:
n_show = 20
mask = (y_true < n_show) & (y_pred < n_show)
cm   = confusion_matrix(y_true[mask], y_pred[mask])

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=[f'C{i}' for i in range(n_show)],
            yticklabels=[f'C{i}' for i in range(n_show)])
ax.set_title('Confusion Matrix — First 20 Flower Classes', fontsize=13)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()
print('[OK] Confusion matrix saved')

## PER - CLASS ACCURACY BAR CHART 

In [ ]:
per_class_acc = []
for c in range(CONFIG['NUM_CLASSES']):
    mask_c = y_true == c
    if mask_c.sum() > 0:
        per_class_acc.append((y_pred[mask_c] == c).mean())
    else:
        per_class_acc.append(0.0)

per_class_acc = np.array(per_class_acc)

fig, ax = plt.subplots(figsize=(20, 5))
colors = ['#4CAF50' if a >= 0.7 else '#FF5722' if a < 0.4 else '#FFC107'
          for a in per_class_acc]
ax.bar(range(102), per_class_acc, color=colors, width=0.8)
ax.axhline(per_class_acc.mean(), color='navy', ls='--',
           label=f'Mean: {per_class_acc.mean()*100:.1f}%')
ax.set_title('Per-Class Accuracy (Green≥70%, Yellow=40-70%, Red<40%)', fontsize=13)
ax.set_xlabel('Class ID'); ax.set_ylabel('Accuracy')
ax.legend(); ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig('/kaggle/working/per_class_accuracy.png', dpi=150)
plt.show()
print(f'Classes with >70% accuracy: {(per_class_acc>=0.7).sum()}/102')

## SAMPLE PREDICTIONS GRID

In [ ]:
x_batch, y_batch = next(iter(test_gen))
preds_batch      = model.predict(x_batch[:12], verbose=0)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Sample Test Predictions', fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    true_lbl = int(np.argmax(y_batch[i]))
    pred_lbl = int(np.argmax(preds_batch[i]))
    conf     = preds_batch[i][pred_lbl] * 100
    is_right = true_lbl == pred_lbl
    color    = '#2E7D32' if is_right else '#C62828'
    result   = '✓' if is_right else '✗'
    ax.imshow(x_batch[i])
    ax.set_title(
        f'{result} True:{true_lbl} → Pred:{pred_lbl}\nConf: {conf:.1f}%',
        color=color, fontsize=9, fontweight='bold',
    )
    ax.axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/sample_predictions.png', dpi=150)
plt.show()
print('[OK] Sample predictions saved')

## SAVE MODEL

In [ ]:
model.save('/kaggle/working/flower_inceptionv3_final.keras')
model.export('/kaggle/working/flower_inceptionv3_savedmodel')
print('[OK] Model saved:')
print('     /kaggle/working/flower_inceptionv3_final.keras  (native Keras)')
print('     /kaggle/working/flower_inceptionv3_savedmodel   (SavedModel/TFLite)')


## FINAL SUMMARY

In [ ]:
log_df = pd.read_csv('/kaggle/working/training_log.csv')

best_val_acc = log_df['val_accuracy'].max()
best_epoch   = log_df['val_accuracy'].idxmax() + 1

print('\n' + '='*65)
print('🌸 IMPROVED PROJECT — Final Summary')
print('='*65)
print(f'  Model             : InceptionV3 (ImageNet → Transfer Learning)')
print(f'  Loss Function     : Hierarchical Cost-Sensitive CE + Label Smoothing')
print(f'                      (Family Penalty λ={CONFIG["FAMILY_PENALTY"]}, ε={CONFIG["LABEL_SMOOTH"]})')
print(f'  Classes           : {CONFIG["NUM_CLASSES"]}')
print(f'')
print(f'  Split (REVERSED):')
print(f'    Train images    : {(df.split=="train").sum()}  (was 1020)')
print(f'    Val images      : {(df.split=="val").sum()}')
print(f'    Test images     : {(df.split=="test").sum()}   (was 6149)')
print(f'')
print(f'  Phase 1 Epochs    : {len(history1.history["loss"])}')
print(f'  Phase 2 Epochs    : {len(history2.history["loss"])}')
print(f'  Unfrozen layers   : {CONFIG["UNFREEZE_LAYERS"]}')
print(f'  Best Val Acc      : {best_val_acc*100:.2f}% (epoch {best_epoch})')
print(f'')
print(f'  TEST RESULTS:')
print(f'    Accuracy        : {accuracy*100:.2f}%')
print(f'    Precision       : {precision*100:.2f}%')
print(f'    Recall          : {recall*100:.2f}%')
print(f'    F1 Score        : {f1*100:.2f}%')
print(f'    Loss            : {test_results[0]:.4f}')
print('='*65)
print('\nOutput files in /kaggle/working:')
for f in sorted(os.listdir('/kaggle/working')):
    fp = os.path.join('/kaggle/working', f)
    if os.path.isfile(fp):
        size_kb = os.path.getsize(fp) / 1024
        print(f'  • {f:<45} {size_kb:>8.1f} KB')